# **Carga de datos**

In [19]:
import os

base_dir = './cats_and_dogs_small'

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Directorio con las imágenes de training
train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')

# Directorio con las imágenes de validación
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

# Directorio con las imágenes de test
test_cats_dir = os.path.join(test_dir, 'cats')
test_dogs_dir = os.path.join(test_dir, 'dogs')

# **Data augmentation**

Para este colab, voy a aplicar algunas de las técnicas de *Data Augmentation*, esto se consigue con la instancia de *ImageDataGenerator* de la siguiente manera.

In [20]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))
validation_generator = validation_datagen.flow_from_directory(validation_dir,
                                                                batch_size=20,
                                                                class_mode='binary',
                                                                target_size=(150, 150))
test_generator = test_datagen.flow_from_directory(test_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


# **Creación del modelo**

El modelo será el mismo que el de el colab *entrenar_datos_reales.ipynb*.

In [21]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

# Primera capa
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

# Segunda capa
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

# Tercera capa
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

# Cuarta capa
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

# Capa de aplanamiento
model.add(Flatten())

# Capa densa
model.add(Dense(512, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [25]:
from tensorflow.keras.optimizers import RMSprop

model.compile(optimizer=RMSprop(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['acc'])

In [27]:
%pip install scipy

model.fit(train_generator, epochs=30, 
          validation_data=validation_generator, 
          steps_per_epoch=100, 
          validation_steps=50, 
          verbose=2)

594.92s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 10.3 MB/s  0:00:03m0:00:0100:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Epoch 1/30
100/100 - 21s - 209ms/step - acc: 0.5075 - loss: 0.7077 - val_acc: 0.5010 - val_loss: 0.6917
Epoch 2/30
100/100 - 19s - 195ms/step - acc: 0.5170 - loss: 0.6954 - val_acc: 0.5000 - val_loss: 0.6942
Epoch 3/30
100/100 - 16s - 157ms/step - acc: 0.5155 - loss: 0.6931 - val_acc: 0.4950 - val_loss: 0.6925
Epoch 4/30
100/100 - 16s - 164ms/step - acc: 0.5120 - loss: 0.6939 - val_acc: 0.5000 - val_loss: 0.6926
Epoch 5/30
100/100 - 15s - 153ms/step - acc: 0.5420 - loss: 0.6892 - val_acc: 0.5320 - val_loss: 0.6914
Epoch 6/30
100/100 - 15s - 149ms/step - acc: 0.5295 - loss: 0.6901 - val_acc: 0.5380 - val_loss: 0.6906
Epoch 7/30
100/100 - 22s - 222ms/step - acc: 0.5335 - loss: 0.6900 - val_acc: 0.5310 - val_loss: 0.6907
Epoch 8/30
100/100 - 17s - 168ms/step - acc: 0.5510 - loss: 0.6846 - val

In [28]:
test_loss, test_acc = model.evaluate(test_generator)
print('test acc:', test_acc)

50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - acc: 0.6420 - loss: 0.6288
test acc: 0.6420000195503235
